# Cuts out the groups from the images

**WARNING Execute "match_lables_with_gt.ipynb" beforehand!**

In [2]:
from pathlib import Path

from PIL import Image
import pandas as pd
from utils import PROJECT_ROOT, build_group_boxes, SPLITS

In [3]:
boxes: dict[str, pd.DataFrame] = {k:None for k in SPLITS}#type:ignore
for key in boxes.keys():
    bb = pd.read_csv(PROJECT_ROOT/f"data/bounding_boxes_with_labels/{key}.csv")
    bb["frame_name"] = bb["flight"].astype(str)+"_"+bb["frame"].astype(str)
    boxes[key] = bb
boxes.keys()

dict_keys(['train', 'val', 'test'])

In [4]:
boxes["test"]

,flight,frame,class_id,species,txt_x1,txt_y1,txt_x2,txt_y2,txt_h,txt_w,old_class_id,frame_name
0,55,4522,b,Cervus elaphus (Red deer),0,479,36,522,42,36,7,55_4522
1,55,4532,b,Cervus elaphus (Red deer),17,504,62,545,40,44,7,55_4532
2,55,4542,b,Cervus elaphus (Red deer),48,533,96,571,37,48,7,55_4542
3,55,4542,b,Cervus elaphus (Red deer),138,533,188,578,45,49,7,55_4542
4,55,4554,b,Cervus elaphus (Red deer),61,523,103,563,39,42,7,55_4554
...,...,...,...,...,...,...,...,...,...,...,...,...
1573,309,26538,c,Capreolus capreolus (Roe deer),186,468,223,502,33,36,21,309_26538
1574,309,26580,c,Capreolus capreolus (Roe deer),319,474,357,508,34,37,21,309_26580
1575,309,26639,c,Capreolus capreolus (Roe deer),509,482,540,516,33,30,21,309_26639
1576,309,26701,c,Capreolus capreolus (Roe deer),693,487,733,525,37,39,21,309_26701


In [ ]:
output_root = PROJECT_ROOT / "data" / "cut_images" / "unclassified"
output_root.mkdir(parents=True, exist_ok=True)

total_saved = 0

for split in SPLITS:
    split_dir = output_root #/ split # will no longer split the outputs, as the default split is nonsense
    split_dir.mkdir(parents=True, exist_ok=True)

    split_boxes = boxes[split]
    for frame_name in split_boxes["frame_name"].unique():
        frame_boxes = split_boxes[split_boxes["frame_name"] == frame_name]
        if frame_boxes.empty:
            continue

        image_path = PROJECT_ROOT / "data" / "rgb_images" / split / f"{frame_name}.jpg"
        if not image_path.exists():
            print(f"Skipping missing image: {image_path}")
            continue

        groups = build_group_boxes(frame_boxes,0.5, min_size=200, eps_scale=4)
        if groups.empty:
            continue

        image = Image.open(image_path).convert("RGB")
        img_w, img_h = image.size

        try:
            flight_id, frame_id = frame_name.rsplit("_", 1)
        except ValueError:
            print(f"Skipping malformed frame_name: {frame_name}")
            continue

        for _, group_row in groups.iterrows():
            x1 = int(max(0, round(group_row["group_x1"])))
            y1 = int(max(0, round(group_row["group_y1"])))
            x2 = int(min(img_w, round(group_row["group_x1"] + group_row["group_w"])))
            y2 = int(min(img_h, round(group_row["group_y1"] + group_row["group_h"])))

            if x2 <= x1 or y2 <= y1:
                continue

            cluster_id = int(group_row["cluster_id"])
            species_counts = group_row["species_in_cluster"]
            species_suffix = "".join([f"{count}{species}" for species, count in species_counts])

            out_name = f"{flight_id}_{frame_id}_g{cluster_id}_{species_suffix}.jpg"
            out_path = split_dir / out_name

            crop = image.crop((x1, y1, x2, y2))
            crop.save(out_path, format="JPEG", quality=95)
            total_saved += 1

print(f"Saved {total_saved} grouped cutouts to: {output_root}")

Saved 6186 grouped cutouts to: C:\Users\jojog\Documents\School\Semester_4\CIV4\CIV4-WhoS-Where\data\cut_images\unclassified
